# Notebook 07: BM25 Ranked Retrieval

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
**Okapi BM25** (Best Matching 25) is the state-of-the-art probabilistic lexical retrieval algorithm. It improves upon TF-IDF in two fundamental ways:
1. **Term Frequency Saturation ($k_1$)**: Diminishing returns for repeated occurrences of a keyword.
2. **Document Length Normalization ($b$)**: Penalizes verbose documents without penalizing documents that are naturally dense.

**Mathematical Formula:**
$$\text{Score}(D, Q) = \sum_{t \in Q} \text{IDF}(t) \cdot \frac{f(t, D) \cdot (k_1 + 1)}{f(t, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

$$\text{IDF}(t) = \ln\left(\frac{N - n(t) + 0.5}{n(t) + 0.5} + 1\right)$$

**Core Objectives:**
1. Instantiate `BM25Retriever` from `src.retrieval.bm25` on the 698-restaurant corpus.
2. Deconstruct the score of a document term-by-term using `explain_score()`.
3. Analyze parameter sensitivity ($k_1$ and $b$).
4. Compare BM25 rankings side-by-side with TF-IDF.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor
from src.indexing.inverted_index import InvertedIndex
from src.retrieval.bm25 import BM25Retriever
from src.retrieval.tfidf import TFIDFRetriever

print(f"Project root resolved: {PROJECT_ROOT}")

## 1. Load Data and Initialize BM25 Model

In [ ]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
docs = converter.convert_all(records)
corpus = converter.get_corpus()

preprocessor = TextPreprocessor()
tokenized_corpus = preprocessor.preprocess_documents(corpus)

index = InvertedIndex()
index.build(tokenized_corpus)

# Standard BM25 parameters: k1=1.5, b=0.75
bm25 = BM25Retriever(index, preprocessor, k1=1.5, b=0.75)
print(f"BM25 Retriever ready across {index.num_documents} documents (avgdl={index.avg_doc_length:.2f} tokens).")

## 2. Execute BM25 Query and Inspect Top Results

In [ ]:
query = "traditional cambodian noodle soup"
bm25_results = bm25.search(query, top_k=5)

print(f"Top 5 BM25 Results for: '{query}'\n")
for r in bm25_results:
    rec = converter.get_record(r["doc_id"])
    print(f"Rank {r['rank']} [BM25 Score: {r['score']:.4f}] -> [{r['doc_id']}] {rec['name']}")
    print(f"   Category: {rec['category']} | Subcategories: {rec['sub_category']}")
    print(f"   Address:  {rec['address']}\n")

## 3. Mathematical Score Breakdown (`explain_score`)

Inspect the exact contribution of each term towards the final score of the top-ranked document.

In [ ]:
top_doc_id = bm25_results[0]["doc_id"]
explanation = bm25.explain_score(query, top_doc_id)

print(f"=== BM25 Score Explanation for Doc {top_doc_id} ===")
print(f"Document Length        : {explanation['doc_length']} tokens")
print(f"Average Document Length : {explanation['avg_doc_length']} tokens")
print(f"Length Norm Factor     : {explanation['length_norm_factor']}")
print(f"Total BM25 Score       : {explanation['total_bm25_score']}\n")

terms_df = pd.DataFrame(explanation["terms"])
print(terms_df.to_string(index=False))

## 4. Hyperparameter Sensitivity Analysis ($b$ and $k_1$)

Observe how varying length normalization parameter $b$ affects scores.

In [ ]:
# Compare b=0.0 (no length normalization) vs b=0.75 (standard) vs b=1.0 (strict)
bm25_b0 = BM25Retriever(index, preprocessor, k1=1.5, b=0.0)
bm25_b75 = BM25Retriever(index, preprocessor, k1=1.5, b=0.75)
bm25_b100 = BM25Retriever(index, preprocessor, k1=1.5, b=1.0)

res_b0 = bm25_b0.search(query, top_k=3)
res_b75 = bm25_b75.search(query, top_k=3)
res_b100 = bm25_b100.search(query, top_k=3)

print("Top Document with b=0.0  :", converter.get_record(res_b0[0]["doc_id"])["name"], f"(Score: {res_b0[0]['score']:.2f})")
print("Top Document with b=0.75 :", converter.get_record(res_b75[0]["doc_id"])["name"], f"(Score: {res_b75[0]['score']:.2f})")
print("Top Document with b=1.0  :", converter.get_record(res_b100[0]["doc_id"])["name"], f"(Score: {res_b100[0]['score']:.2f})")

## 5. Side-by-Side Comparison: BM25 vs. TF-IDF

In [ ]:
tfidf = TFIDFRetriever(index, preprocessor)
test_queries = ["italian pizza pasta", "craft beer burger bbq", "street 125 noodles"]

for q in test_queries:
    bm25_top = bm25.search(q, top_k=3)
    tfidf_top = tfidf.search(q, top_k=3)
    
    print(f"\n================ Query: '{q}' ================")
    print("BM25 Top 3:")
    for r in bm25_top:
        name = converter.get_record(r["doc_id"])["name"]
        print(f"  [{r['rank']}] {name} (Score: {r['score']:.4f})")
        
    print("TF-IDF Top 3:")
    for r in tfidf_top:
        name = converter.get_record(r["doc_id"])["name"]
        print(f"  [{r['rank']}] {name} (Cosine: {r['score']:.4f})")

## 6. Observations & Takeaways
- **BM25** provides superior term frequency saturation and document length normalization compared to raw TF-IDF.
- It handles exact keywords (e.g. street numbers like *'125'*, specific dishes like *'noodles'*) with high accuracy.
- **Next Step (Phase 4)**: Implement Dense Semantic Vector Search using `multilingual-e5-small` and FAISS, before fusing both in **Phase 5 (Hybrid Search)**!